In [1]:
import sys
!{sys.executable} -m pip install -U google-genai python-dotenv ipywidgets python-docx


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
load_dotenv() 
gemini_key = os.getenv("GEMINI_API_KEY")
print(gemini_key[:6])


AIzaSy


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

LABEL_W = "80px"
FIELD_W = "700px"

common_style = {"description_width": LABEL_W}
text_layout = widgets.Layout(width=FIELD_W)
area_layout = widgets.Layout(width=FIELD_W, height="160px")

topic_w = widgets.Text(description="보고서 제목", layout=widgets.Layout(width="300px"))
purpose_w   = widgets.Text(description="보고서 목적", layout=widgets.Layout(width="600px"))
requirements_w = widgets.Textarea(description="요구사항", layout=widgets.Layout(width="600px", height="100px"), placeholder="예: 1. 2.")

btn = widgets.Button(description="확인", button_style="primary")
btn_box = widgets.HBox([btn], layout=widgets.Layout(justify_content="center", width="700px"))
out = widgets.Output()

result = {}  # 입력값 저장용

def on_click(_):
    result["topic"] = topic_w.value
    result["purpose"] = purpose_w.value
    result["requirements"] = requirements_w.value
    with out:
        clear_output()
        print("입력 완료")
        print(result)

btn.on_click(on_click)

display(topic_w, purpose_w, requirements_w, btn_box, out)


Text(value='', description='보고서 제목', layout=Layout(width='300px'))

Text(value='', description='보고서 목적', layout=Layout(width='600px'))

Textarea(value='', description='요구사항', layout=Layout(height='100px', width='600px'), placeholder='예: 1. 2.')

Output()

In [21]:
import os
from google import genai
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# 상태 저장
toc_state = {"toc": "", "final_toc": ""}

def get_inputs(default_topic="미정(보고서 제목)", default_purpose="미정(보고서 목적)", default_requirements="없음"):
    r = globals().get("result", {}) or {}

    def pick(key, widget_name):
        w = globals().get(widget_name, None)
        return (r.get(key) or (getattr(w, "value", "") if w else "") or "").strip()

    topic = pick("topic", "topic_w") or default_topic
    purpose = pick("purpose", "purpose_w") or default_purpose
    requirements = pick("requirements", "requirements_w") or default_requirements
    return topic, purpose, requirements

def make_prompt(mode, topic, purpose, requirements, toc=None, feedback=None):
    """mode: 'toc' | 'revise'"""
    if mode == "toc":
        return f"""
너는 컨설팅 보고서 작성 전문가다.
아래 입력을 바탕으로 '보고서 목차(TOC)'를 한국어로 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[출력 요구사항]
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 최소 5개 장(Chapter) 이상
- 장(1,2,3...)마다 소절(1.1, 1.2...)은 필요시 생성
- 불필요한 설명문 없이 목차만 출력
""".strip()

    if mode == "revise":
        return f"""
너는 컨설팅 보고서 편집자다.
아래 기존 목차와 사용자 수정 지시를 반영하여 '개정 목차'를 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[기존 목차]
{toc}

[목차 수정]
{feedback}

[출력 요구사항]
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 불필요한 설명문 없이 개정 목차만 출력
- #이나 * 기호 절대 사용금지지
""".strip()

    raise ValueError("mode는 'toc' 또는 'revise'여야 합니다.")

def toc_ui(model_name="gemini-2.0-flash", width="800px"):
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY 환경변수가 없습니다. (.env 로드 또는 OS 환경변수 설정 필요)")

    topic, purpose, requirements = get_inputs()
    client = genai.Client(api_key=api_key)

    # 최초 목차 생성
    resp = client.models.generate_content(
        model=model_name,
        contents=make_prompt("toc", topic, purpose, requirements)
    )
    toc_state["toc"] = (resp.text or "").strip()
    toc_state["final_toc"] = toc_state["toc"]

    out = widgets.Output()

    def render():
        with out:
            clear_output()
            display(Markdown("### 생성된 목차\n\n```text\n" + toc_state["final_toc"] + "\n```"))

    feedback_w = widgets.Textarea(
        description="목차 수정",
        placeholder="예: 2장을 '시장/정책 환경'으로 변경, 3.2에 리스크 관리 추가",
        style={"description_width": "80px"},
        layout=widgets.Layout(width=width, height="120px")
    )

    apply_btn = widgets.Button(description="반영", button_style="primary")
    btn_box = widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center", width=width))

    def on_apply(_):
        feedback = (feedback_w.value or "").strip()
        if not feedback:
            toc_state["final_toc"] = toc_state["toc"]
        else:
            r = client.models.generate_content(
                model=model_name,
                contents=make_prompt("revise", topic, purpose, requirements, toc=toc_state["toc"], feedback=feedback)
            )
            toc_state["final_toc"] = (r.text or "").strip()

        # 최종 목차를 화면에 반영
        render()

    apply_btn.on_click(on_apply)

    render()
    display(out, feedback_w, btn_box)

# 실행
toc_ui(model_name="gemini-2.0-flash")


Output()

Textarea(value='', description='목차 수정', layout=Layout(height='120px', width='800px'), placeholder="예: 2장을 '시장/…

In [ ]:
# #5번 코드
import os
import re
from google import genai

# 전역 누적 변수
final_report = f"{topic}\n\n"
final_summary = ""
final_sources = ""


def extract_chapters(final_toc: str):
    chapters = []
    for line in (final_toc or "").splitlines():
        s = line.strip()
        # '1. 서론' 형태만 추출 (1.1 제외)
        if re.match(r"^\d+\.\s+\S+", s):
            chapters.append(s)
    return chapters


def make_prompt_for_report(topic, purpose, requirements, final_toc, chapter_title, summary_context=""):
    # 파싱 안정화를 위해 블록 태그 강제
    return f"""
너는 컨설팅 보고서 작성 전문가로서, 아래와 같은 지침을 따르며 보고서를 작성해야 한다.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 보고서 요구사항: {requirements}
- 보고서 목차: {final_toc}
- 해당 장 제목: {chapter_title}
- 이전 내용: {summary_context}

[출력 형식 - 반드시 그대로 출력]
[REPORT]
(해당 장 보고서 본문)
[/REPORT]

[SUMMARY]
(해당 장 핵심 요약 3문장)
[/SUMMARY]

[SOURCES]
(해당 장에서 사용/인용한 출처 목록. MLA)
[/SOURCES]

[출력 요구사항]
1. REPORT : 주어진 목차의 해당 장 제목에 맞게 본문을 자세히 작성하라.
   - 단순 설명을 지양하고, 원인·구조·배경·영향·시사점의 관점에서 심층적이고 전문적인 분석을 수행할 것.
   - 해당 장의 주요 내용과 주제를 명확히 이해하고, 주제에 대한 정확한 분석과 객관적인 사실을 제공하라.
   - 관련 데이터(통계, 조사결과, 시장 동향 등)와 사례(기존 연구, 산업 사례, 비교 분석 등)를 적극 활용하여 주장을 뒷받침하라.
   - 이전 내용을 참고하여 논리적 흐름을 자연스럽게 이어가도록 하라.
   - 검증된 사실 내에서 최대한 많은 정보를 제공하여라.
   - 내용이 목적에 부합하도록 작성하여라.
   - 주의: 장 제목과 소제목은 ##이 없는(예: "1. 서론" "1.1 정의")의 형태로 작성한다.
   - 주의: 소제목 아래는 절대로 빈줄 공백줄을 사용하지 않는다. 
   - 엔터,빈줄과 공백줄은 절대 사용하지 않는다.
   - 주의: 문자 *는 절대 사용하지 않으며, 문자 양옆에 강조하기 위한 기호는 사용하지 않는다. 
   - 나열시 줄바꿈은 허용하며, -기호를 사용하여 리스트 형식으로 나열한다.

2. SUMMARY: 해당 장의 핵심 내용을 2문장으로 간략하게 요약하라.
   - 주요 논점과 결론과 핵심적인 세부사항을 포함하라.

3. SOURCES: REPORT를 작성하면서 사용한 모든 출처와 관련된 정보들을 나열하라.
   - 논문, 보고서, 연구 결과, 웹사이트 등 다양한 출처를 명확히 구분하여 기록하라.
   - mla 방식으로 출처 표시의 일관성을 유지하라.

""".strip()


def parse_response_blocks(text: str):
    def between(t: str, a: str, b: str) -> str:
        if a not in t or b not in t:
            return ""
        return t.split(a, 1)[1].split(b, 1)[0].strip()

    report = between(text or "", "[REPORT]", "[/REPORT]")
    summary = between(text or "", "[SUMMARY]", "[/SUMMARY]")
    sources = between(text or "", "[SOURCES]", "[/SOURCES]")
    return report, summary, sources


def generate_report_from_toc(final_toc, topic, purpose, requirements, api_key, model="gemini-2.5-pro"):
    global final_report, final_summary, final_sources

    # 매 실행 시 초기화(중복 누적 방지)
    final_report = ""
    final_summary = ""
    final_sources = ""

    client = genai.Client(api_key=api_key)

    # 장(1.,2.,3. ...)만 추출
    chapters = extract_chapters(final_toc)

    # 다음 챕터 프롬프트에 넣을 "누적 요약 컨텍스트"
    summary_context = ""

    for chapter_title in chapters:
        prompt = make_prompt_for_report(
            topic, purpose, requirements, final_toc, chapter_title, summary_context
        )

        response = client.models.generate_content(model=model, contents=prompt)
        r, s, src = parse_response_blocks(getattr(response, "text", "") or "")

        # 최종 산출물 누적
        final_report += f"{r}\n\n"
        final_summary += f" {s}"
        final_sources += f"{src}"

        # 요약 컨텍스트 누적 (다음 챕터가 참고)
        summary_context += f"- {chapter_title}: {s}"

    return final_report, final_summary, final_sources, summary_context


# ===== 실행 =====
topic, purpose, requirements = get_inputs()

final_report, final_summary, final_sources, summary_context = generate_report_from_toc(
    final_toc=toc_state["final_toc"],
    topic=topic,
    purpose=purpose,
    requirements=requirements,
    api_key=gemini_key,
    model="gemini-2.5-pro"
)

print("==== final_report (first 500 chars) ====")
print(final_report[:500])

print("\n==== final_sources (first 500 chars) ====")
print(final_sources[:500])

print("\n==== summary_context (first 500 chars) ====")
print(summary_context[:500])


==== final_report (first 500 chars) ====
1. 서론
1.1. 연구 배경 및 필요성
미국 캘리포니아 주는 2045년까지 100% 무탄소 전력 공급을 목표로 하는 법안(SB 100)을 필두로 전 세계적인 탈탄소 에너지 전환을 선도하고 있다. 이러한 정책 기조 아래 태양광, 풍력 등 재생에너지 발전 비중이 급격히 증가하고 있으며, 이는 기존 전력 계통의 안정성에 새로운 도전 과제를 제기한다. 특히, 일조량에 따라 발전량이 급변하는 태양광의 특성은 이른바 '덕 커브(Duck Curve)' 현상을 심화시켜, 저녁 시간대 전력 수요가 급증할 때 유연하고 신속하게 대응할 수 있는 발전원의 필요성을 증대시키고 있다. 전통적으로 이러한 역할을 수행해 온 것이 바로 천연가스를 연료로 사용하는 가스 피커(Peaker) 발전소이다. 가스 피커 발전소는 빠른 기동 시간과 출력 조절 능력을 바탕으로 재생에너지의 간헐성을 보완하고 전력망의 주파수와 전압을 안정적으로 유지하는 데 필수적인 역할을 담당해왔다. 그러나 캘리포니아 주의 강력한 탈탄소 정책은

==== final_sources (first 500 chars) ====
California Energy Commission. "2021 Integrated Energy Policy Report." *California Energy Commission*, Publication Number: CEC-100-2021-001-CMF, 2021, www.energy.ca.gov/datareports/reports/integrated-energy-policy-report/2021-integrated-energy-policy-report.
California ISO. "Annual Report on Market Issues and Performance." *California ISO*, 2022, www.caiso.com/Documents/2022-Annual-Report-on-Market-Issues-and-Per

In [23]:
from docx import Document
import os
import re

def sanitize_filename(name: str, default="report"):
    name = (name or "").strip() or default
    # Windows 금지 문자 제거
    name = re.sub(r'[\\/:*?"<>|]', "_", name)
    # 너무 길면 잘라내기(옵션)
    return name[:150]

def get_download_path(file_name="report.docx"):
    if os.name == "nt":  # Windows
        download_path = os.path.join(os.environ.get("USERPROFILE", ""), "Downloads")
    else:  # macOS, Linux
        download_path = os.path.join(os.environ.get("HOME", ""), "Downloads")

    if not download_path or not os.path.isdir(download_path):
        # Downloads가 없으면 현재 폴더로 fallback
        download_path = os.getcwd()

    os.makedirs(download_path, exist_ok=True)
    return os.path.join(download_path, file_name)

def add_multiline_text(doc: Document, text: str):
    for line in (text or "").splitlines():
        if line.strip() == "":
            doc.add_paragraph("")  # 빈 줄 유지
        else:
            doc.add_paragraph(line)

def save_report_and_sources(final_report, final_sources, topic):
    file_name = f"{sanitize_filename(topic)}.docx"
    file_path = get_download_path(file_name)

    doc = Document()

    doc.add_heading("Report", level=1)
    add_multiline_text(doc, final_report)

    doc.add_heading("Sources", level=1)
    add_multiline_text(doc, final_sources)

    doc.save(file_path)
    print(f"파일이 저장되었습니다: {file_path}")

save_report_and_sources(final_report, final_sources, topic)


파일이 저장되었습니다: C:\Users\mphk0\Downloads\미국 캘리포니아 가스 피커 전망.docx
